# 01. Python and Regular Expressions Preparation

This notebook builds practical Python foundations for the recruitment assistant using synthetic resume data only. Each section produces a small, inspectable result.

## 1. Introduction

The Week-2 goal is to prepare for resume parsing: read text, identify fields, and organize extracted values into tabular data.

## 2. Project Context

A future parser may receive a resume as TXT, PDF, or DOCX. Month 1 uses TXT examples and regular expressions; it does not claim to be a production parser.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from pathlib import Path
import re
import pandas as pd

print(ROOT)
print("Synthetic data only: Candidate_001")

C:\Users\katri\OneDrive\Desktop\COLLEGE\Internship\AI-Powered-Recruitment-Assistant
Synthetic data only: Candidate_001


## 3. Python Variables and Data Structures

In [2]:
candidate_label = "Candidate_001"
role = "Data Analyst"
skills = ["Python", "SQL", "pandas"]
contact = {"email": "candidate001@example.test", "phone": "000-555-0001"}

print(candidate_label, role)
print("Skills:", skills)
print("Email from dictionary:", contact["email"])

Candidate_001 Data Analyst
Skills: ['Python', 'SQL', 'pandas']
Email from dictionary: candidate001@example.test


## 4. String Operations

In [3]:
raw_title = "  Synthetic Data Analyst Resume  "
print("strip:", raw_title.strip())
print("lower:", raw_title.lower())
print("replace:", raw_title.strip().replace("Resume", "Profile"))
print("contains analyst:", "analyst" in raw_title.lower())

strip: Synthetic Data Analyst Resume
lower:   synthetic data analyst resume  
replace: Synthetic Data Analyst Profile
contains analyst: True


## 5. Reading a TXT Resume from File

In [4]:
resume_path = ROOT / "data" / "resumes" / "resume_001.txt"
resume_text = resume_path.read_text(encoding="utf-8")
# Add synthetic contact lines for this extraction exercise; no real contact data is used.
resume_text += "\nEmail: candidate001@example.test\nPhone: 000-555-0001\n"
print(resume_text)

Synthetic Data Analyst candidate with skills Python;SQL and 2 years of experience.

Email: candidate001@example.test
Phone: 000-555-0001



## 6. Writing Resume Text to a File

In [5]:
demo_output = ROOT / "data" / "processed" / "notebook_resume_copy.txt"
demo_output.write_text(resume_text, encoding="utf-8")
print("Wrote:", demo_output.relative_to(ROOT))
print("Characters written:", len(demo_output.read_text(encoding="utf-8")))

Wrote: data\processed\notebook_resume_copy.txt
Characters written: 137


## 7. Basic Text Processing

In [6]:
compact_text = " ".join(resume_text.split())
print("Original characters:", len(resume_text))
print("Compact preview:", compact_text[:180])

Original characters: 137
Compact preview: Synthetic Data Analyst candidate with skills Python;SQL and 2 years of experience. Email: candidate001@example.test Phone: 000-555-0001


## 8. Regular Expressions Introduction

In [7]:
sample = "Email candidate001@example.test; phone 000-555-0001; Python and SQL; experience 2 years."
print(re.findall(r"\b[A-Za-z]+\b", sample)[:12])
print(re.findall(r"\b\d+\b", sample))

['Email', 'example', 'test', 'phone', 'Python', 'and', 'SQL', 'experience', 'years']
['000', '555', '0001', '2']


## 9. Extract Email Address

In [8]:
email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
emails = re.findall(email_pattern, resume_text)
print("Emails:", emails)

Emails: ['candidate001@example.test']


## 10. Extract Phone Number

In [9]:
phone_pattern = r"\b\d{3}[- ]\d{3}[- ]\d{4}\b"
phones = re.findall(phone_pattern, resume_text)
print("Phones:", phones)

Phones: ['000-555-0001']


## 11. Extract Skills

In [10]:
skill_dictionary = ["Python", "SQL", "pandas", "NumPy", "scikit-learn", "NLP", "Docker"]
found_skills = [skill for skill in skill_dictionary if re.search(rf"\b{re.escape(skill)}\b", resume_text, flags=re.I)]
print("Detected skills:", found_skills)

Detected skills: ['Python', 'SQL']


## 12. Extract Years of Experience

In [11]:
experience_match = re.search(r"(\d+)\s+years?", resume_text, flags=re.I)
experience_years = int(experience_match.group(1)) if experience_match else None
print("Experience years:", experience_years)

Experience years: 2


## 13. Create a Structured Resume Dictionary

In [12]:
structured_resume = {
    "resume_id": "RES_001",
    "candidate_label": candidate_label,
    "email": emails[0] if emails else None,
    "phone": phones[0] if phones else None,
    "skills": ";".join(found_skills),
    "experience_years": experience_years,
    "resume_text": compact_text,
}
structured_resume

{'resume_id': 'RES_001',
 'candidate_label': 'Candidate_001',
 'email': 'candidate001@example.test',
 'phone': '000-555-0001',
 'skills': 'Python;SQL',
 'experience_years': 2,
 'resume_text': 'Synthetic Data Analyst candidate with skills Python;SQL and 2 years of experience. Email: candidate001@example.test Phone: 000-555-0001'}

## 14. Convert Multiple Resume Records into a pandas DataFrame

In [13]:
records = [structured_resume, {**structured_resume, "resume_id": "RES_002", "candidate_label": "Candidate_002", "email": "candidate002@example.test"}]
resume_frame = pd.DataFrame(records)
resume_frame

,resume_id,candidate_label,email,phone,skills,experience_years,resume_text
0,RES_001,Candidate_001,candidate001@example.test,000-555-0001,Python;SQL,2,Synthetic Data Analyst candidate with skills P...
1,RES_002,Candidate_002,candidate002@example.test,000-555-0001,Python;SQL,2,Synthetic Data Analyst candidate with skills P...


## 15. Inspect DataFrame

In [14]:
print("Shape:", resume_frame.shape)
print("Columns:", resume_frame.columns.tolist())
print(resume_frame[["candidate_label", "skills", "experience_years"]].head())

Shape: (2, 7)
Columns: ['resume_id', 'candidate_label', 'email', 'phone', 'skills', 'experience_years', 'resume_text']
  candidate_label      skills  experience_years
0   Candidate_001  Python;SQL                 2
1   Candidate_002  Python;SQL                 2


## 16. Small Practice Exercise

Extract the first capitalized role word from a synthetic sentence. This is a simple exercise, not a complete occupation extractor.

In [15]:
practice_text = "Synthetic ML Engineer candidate works with Python and Docker."
role_match = re.search(r"\b(ML Engineer|Data Analyst|NLP Engineer)\b", practice_text)
print("Practice role:", role_match.group(1) if role_match else "Not found")

Practice role: ML Engineer


## 17. Key Learning Outcomes

- Read and write UTF-8 text files with `pathlib`.
- Use variables, lists, dictionaries, strings, and pandas DataFrames.
- Apply regex patterns to emails, phone numbers, skills, and experience.
- Understand that extraction rules need validation before production use.